# Projeto: Modelo Preditivo de Evolução de Casos de Dengue

**Autor:** Celso Daniel
**Disciplina:** Data Science

## Objetivos
Este projeto utiliza modelos de Árvore de Decisão para prever se pacientes notificados com dengue serão hospitalizados, focando especialmente em comorbidades e sintomas clínicos. O objetivo é criar um modelo de triagem que possa ser utilizado no momento da entrada do paciente no sistema de saúde.

# Preparação dos Dados

## Importando bibliotecas

- **pandas** → manipulação e análise de dados em DataFrames.  
- **numpy** → cálculos numéricos com vetores e matrizes.  
- **seaborn** → visualizações estatísticas com boa estética.  
- **matplotlib.pyplot** → criação de gráficos personalizados.  
- **sklearn** → conjunto de ferramentas para Machine Learning, incluindo algoritmos de classificação, regressão, clustering, redução de dimensionalidade e pré-processamento de dados.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import chi2, f_classif

In [5]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Carregando os dados para um DataFrame

In [7]:
# df = pd.read_csv("dados/DENGBR21_25.csv", sep=";", low_memory=False)
df = pd.read_csv("dados/DENGBR21_25_BALANCEADO.csv", low_memory=False)

In [8]:
print(f"Total de linhas: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")

Total de linhas: 971458
Total de colunas: 121


## Limpeza de Colunas Irrelevantes ou Incompletas

Antes da etapa de modelagem, foi realizada uma auditoria rigorosa de dados ausentes (*missing values*) tanto a nível de colunas (atributos) quanto de linhas (registros). O objetivo foi eliminar o ruído estatístico decorrente do preenchimento negligenciado ou condicional nas fichas do SINAN.

### Critérios de Exclusão de Atributos
Uma triagem automatizada foi aplicada para remover variáveis que se enquadrassem em pelo menos um dos três critérios abaixo:
1. **Atributos com mais de 60% de missing values:** Variáveis cujo preenchimento é extremamente esparso (como datas de exames específicos ou complicações raras) foram descartadas, pois a imputação de dados geraria vieses bizarros.
2. **Atributos de Variância Zero (Constantes):** Colunas que possuíam apenas um valor único para todas as linhas (ex: `TP_NOT` ou `ID_AGRAVO`, que identificam que todos os registros são notificações de dengue) não trazem poder preditivo e foram removidas.
3. **Vazamento de Informação (Data Leakage) e Metadados do Sistema:** Identificadores únicos, códigos de unidade de saúde, datas de digitação/encerramento e variáveis que revelavam o desfecho tardio do paciente (como `EVOLUCAO`, `CLASSI_FIN` e `DT_INTERNA`) foram removidos manualmente para evitar que o modelo fizesse predições baseadas em informações do futuro.

No total, **77 colunas** foram eliminadas do dataset original.

### Impacto da Limpeza na Qualidade da Base
A eficácia desta estratégia de filtragem é evidenciada pela drástica mudança no comportamento estatístico das linhas do dataset, conforme detalhado na tabela abaixo:

| Métrica de Missing por Linha | Base Bruta (Original) | Base Tratada (Pós-Limpeza) |
| :--- | :---: | :---: |
| **Média de Missing por Linha** | 50,16% | 7,91% |
| **Desvio Padrão (std)** | 7,68% | 7,77% |
| **Primeiro Quartil (25%)** | 48,76% | 2,27% |
| **Mediana (50%)** | 51,23% | 6,81% |
| **Terceiro Quartil (75%)** | 53,71% | 13,63% |
| **Máximo de Missing em uma Linha** | 76,03% | 61,36% |

### Análise dos Atributos Remanescentes
Após a filtragem, a base final estabilizou-se em **44 colunas**. Os dados ausentes residuais concentram-se majoritariamente em exames laboratoriais (como `HISTOPA_N` e `IMUNOH_N`, na faixa de 49% de missing). Esse comportamento é epidemiologicamente justificável, visto que nem todo paciente notificado realiza a confirmação via histopatologia ou isolamento viral.

Por outro lado, as variáveis de sintomas clínicos essenciais (como `FEBRE`, `CEFALEIA`, `VOMITO`, `DIABETES`, etc.) apresentaram uma taxa mínima de missing de apenas **1,05%**, garantindo uma matriz rica em sinais clínicos para que a Árvore de Decisão consiga mapear com segurança os padrões de evolução para hospitalização.

In [9]:
def show_missing_stats(df):
  missing_by_row = df.isna().mean(axis=1) * 100
  print("---------- Missing por Linhas ----------")
  print(missing_by_row.describe())
  print("----------------------------------------\n")

  missing_by_col = (df.isna().sum() / df.shape[0]).sort_values(ascending=False)
  print("---------- Missing por Colunas ---------")
  print(missing_by_col)
  print("----------------------------------------\n")

In [10]:
show_missing_stats(df)

---------- Missing por Linhas ----------
count    971458.000000
mean         50.161966
std           7.683479
min           3.305785
25%          48.760331
50%          51.239669
75%          53.719008
max          76.033058
dtype: float64
----------------------------------------

---------- Missing por Colunas ---------
DT_CHIK_S2    0.999989
DT_PRNT       0.999865
DT_CHIK_S1    0.999166
DT_VIRAL      0.995449
DT_OBITO      0.987007
SANGRAM       0.986240
HEMATURA      0.986240
DOENCA_TRA    0.986240
METRO         0.986240
GENGIVO       0.986240
MANI_HEMOR    0.986240
EPISTAXE      0.986240
LACO_N        0.986240
MIGRADO_W     0.986240
FLXRECEBI     0.986240
COMPLICA      0.986240
CON_FHD       0.986240
PLAQ_MENOR    0.986240
EVIDENCIA     0.986240
PLASMATICO    0.986240
PETEQUIAS     0.986240
DT_GRAV       0.986088
NDUPLIC_N     0.984387
RES_CHIKS2    0.984020
RESUL_PRNT    0.983991
CLINC_CHIK    0.982950
GRAV_MIOC     0.972372
GRAV_ORGAO    0.972372
GRAV_AST      0.972371
GRAV_SANG 

In [ ]:
cols_to_remove = ["ID_REGIONA", "ID_UNIDADE", "ID_RG_RESI", "ID_PAIS"
                  , "TP_SISTEMA", "DT_DIGITA", "CS_FLXRET", "TPAUTOCTO",
                   "COUFINF", "COPAISINF", "SEM_NOT", "SEM_PRI", "ANO_NASC", "DT_ENCERRA", "EVOLUCAO" "CLASSI_FIN"]

for col in df.columns:
  if len(df[col].value_counts(dropna=False)) == 1 or df[col].isna().sum() / df.shape[0] > 0.60:
    cols_to_remove.append(col)

print("Colunas que serão removidas:", len(cols_to_remove))
print(cols_to_remove, "\n")

df = df.drop(columns=cols_to_remove)

Colunas que serão removidas: 80
['ID_REGIONA', 'ID_UNIDADE', 'ID_RG_RESI', 'ID_PAIS', 'TP_SISTEMA', 'DT_DIGITA', 'CS_FLXRET', 'TPAUTOCTO', 'COUFINF', 'UF', 'COPAISINF', 'SEM_NOT', 'SEM_PRI', 'ANO_NASC', 'DT_ENCERRA', 'EVOLUCAO', 'DT_INTERNA', 'MUNICIPIO', 'TP_NOT', 'ID_AGRAVO', 'ID_OCUPA_N', 'DT_CHIK_S1', 'DT_CHIK_S2', 'DT_PRNT', 'RES_CHIKS1', 'RES_CHIKS2', 'RESUL_PRNT', 'DT_SORO', 'DT_NS1', 'DT_VIRAL', 'DT_PCR', 'SOROTIPO', 'DT_INTERNA', 'UF', 'MUNICIPIO', 'DOENCA_TRA', 'CLINC_CHIK', 'DT_OBITO', 'ALRM_HIPOT', 'ALRM_PLAQ', 'ALRM_VOM', 'ALRM_SANG', 'ALRM_HEMAT', 'ALRM_ABDOM', 'ALRM_LETAR', 'ALRM_HEPAT', 'ALRM_LIQ', 'DT_ALRM', 'GRAV_PULSO', 'GRAV_CONV', 'GRAV_ENCH', 'GRAV_INSUF', 'GRAV_TAQUI', 'GRAV_EXTRE', 'GRAV_HIPOT', 'GRAV_HEMAT', 'GRAV_MELEN', 'GRAV_METRO', 'GRAV_SANG', 'GRAV_AST', 'GRAV_MIOC', 'GRAV_CONSC', 'GRAV_ORGAO', 'DT_GRAV', 'MANI_HEMOR', 'EPISTAXE', 'GENGIVO', 'METRO', 'PETEQUIAS', 'HEMATURA', 'SANGRAM', 'LACO_N', 'PLASMATICO', 'EVIDENCIA', 'PLAQ_MENOR', 'CON_FHD', 'COMPLIC

In [12]:
show_missing_stats(df)

---------- Missing por Linhas ----------
count    971458.000000
mean          7.916365
std           7.777467
min           0.000000
25%           2.272727
50%           6.818182
75%          13.636364
max          61.363636
dtype: float64
----------------------------------------

---------- Missing por Colunas ---------
HISTOPA_N     0.496823
IMUNOH_N      0.495801
RESUL_VI_N    0.443245
RESUL_SORO    0.436631
RESUL_PCR_    0.416359
RESUL_NS1     0.398053
COMUNINF      0.376302
CS_ESCOL_N    0.160316
CRITERIO      0.036260
CEFALEIA      0.010562
MIALGIA       0.010562
FEBRE         0.010562
ACIDO_PEPT    0.010562
DIABETES      0.010562
LACO          0.010562
DOR_RETRO     0.010562
PETEQUIA_N    0.010562
ARTRALGIA     0.010562
ARTRITE       0.010562
LEUCOPENIA    0.010562
HEPATOPAT     0.010562
AUTO_IMUNE    0.010562
RENAL         0.010562
EXANTEMA      0.010562
NAUSEA        0.010562
VOMITO        0.010562
DOR_COSTAS    0.010562
CONJUNTVIT    0.010562
HEMATOLOG     0.010562
HIPERTENSA

## Cálculo de Intervalos Temporais (Deltas)

Modelos baseados em árvores de decisão não conseguem extrair padrões diretos a partir de variáveis do tipo data/timestamp (como `DT_NOTIFIC`). Para extrair o valor preditivo oculto nesses metadados, transformou-se as datas brutas em variáveis numéricas contínuas de intervalos de tempo (Deltas), mapeando o comportamento do fluxo de atendimento.

### Criação de Novas Variáveis Preditivas

1. **`DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO`:** Calcula o tempo (em dias) que o paciente levou desde o aparecimento dos primeiros sintomas (`DT_SIN_PRI`) até o momento em que o caso foi formalmente notificado no sistema (`DT_NOTIFIC`). 
   * *Justificativa Clínica:* Pacientes que demoram mais a procurar o sistema de saúde ou cujos sintomas demoram a ser correlacionados com dengue podem apresentar maior risco de evolução grave e complicação por falta de manejo clínico oportuno. A distribuição mostrou que a grande maioria das notificações ocorre em até 4 dias após o início dos sintomas.

2. **`DELTA_NOTIFICACAO_INVESTIGACAO` (Temporária):** Calcula o tempo entre a notificação e o início da investigação epidemiológica (`DT_INVEST`). A análise exploratória revelou que em **93,7% dos casos** (`910.975` registros), a investigação inicia no exato mesmo dia ($0$ dias). Por apresentar variância quase nula no cenário global, essa métrica foi utilizada como diagnóstico operacional e descartada para a modelagem.

Após a extração dos deltas preditivos, realizou-se uma limpeza cirúrgica de segurança, eliminando as seguintes colunas remanescentes:
* **`DT_INVEST`, `DT_NOTIFIC` e `DT_SIN_PRI`:** Removidas por se tornarem redundantes após a engenharia de atributos.


In [ ]:
df["DT_NOTIFIC"] = pd.to_datetime(df["DT_NOTIFIC"])
df["DT_SIN_PRI"] = pd.to_datetime(df["DT_SIN_PRI"])
df["DT_INVEST"] = pd.to_datetime(df["DT_INVEST"])

df["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"] = (df["DT_NOTIFIC"] - df["DT_SIN_PRI"]).dt.days
print("------------------------------------------------------------")
print(df["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"].value_counts().head())
print("------------------------------------------------------------\n")

df["DELTA_NOTIFICACAO_INVESTIGACAO"] = df["DT_INVEST"] - df["DT_NOTIFIC"]
print("------------------------------------------------------------")
print(df["DELTA_NOTIFICACAO_INVESTIGACAO"].value_counts().head())
print("------------------------------------------------------------\n")

df = df.drop(columns=["DELTA_NOTIFICACAO_INVESTIGACAO", "DT_INVEST", "DT_NOTIFIC", "DT_SIN_PRI"])

------------------------------------------------------------
DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO
1    155314
2    148873
3    142555
4    108184
0     84909
Name: count, dtype: int64
------------------------------------------------------------

------------------------------------------------------------
DELTA_NOTIFICACAO_INVESTIGACAO
0 days    910975
1 days     12594
2 days      5791
3 days      4554
4 days      3165
Name: count, dtype: int64
------------------------------------------------------------



## Limpeza Geográfica

Para otimizar o desempenho do algoritmo e evitar a maldição da dimensionalidade (excesso de colunas categóricas com alta cardinalidade), realizamos um estudo do comportamento geográfico dos fluxos de atendimento, infecção e residência dos pacientes.

### Insights Operacionais e Validação dos Dados

1. **Município de Notificação vs. Município de Residência:**
   A análise revelou que em **89,4% dos casos** (`869.185` registros), o paciente é notificado e atendido no mesmo município em que reside. Isso demonstra uma forte centralização do atendimento médico local.

2. **UF de Notificação vs. UF de Residência:**
   A consistência a nível estadual é ainda maior: em **99,2% dos casos** (`964.218` registros), o estado onde o caso foi registrado coincide com o estado de residência do cidadão. *(Nota técnica: Foi necessário aplicar um tratamento prévio na coluna `SG_UF_NOT` usando `.str.replace("sp", "35")` para padronizar strings residuais com o código IBGE do estado de São Paulo).*

3. **Município de Notificação vs. Município de Provável Infecção (`COMUNINF`):**
   Aqui encontramos a maior variabilidade: em apenas **55,4% dos casos** (`538.756` registros) o local da infecção coincide com o local do atendimento. Quase metade dos pacientes contraiu a dengue em outro município (seja por viagens, trabalho ou vetores móveis). 
   Os códigos mais comuns de infecção apontam majoritariamente para grandes centros urbanos e focos endêmicos, como o código `355030.0` (Município de São Paulo) com mais de 37 mil ocorrências.

### Justificativa para Remoção das Colunas

Com base nesses achados, as variáveis `ID_MN_RESI`, `SG_UF` e `COMUNINF` foram **descartadas** da base de modelagem pelos seguintes motivos:

* **Redundância Estatística (Multicolinearidade):** Como quase 90% da base apresenta o mesmo valor entre o local de residência e o local de atendimento, manter ambas as colunas faria o modelo de árvore aprender regras repetitivas e redundantes, sem ganho real de informação.
* **Alta Cardinalidade:** Colunas de municípios possuem milhares de códigos únicos. Mantê-las exigiria técnicas como *One-Hot Encoding* massivos, o que inflaria o tamanho da matriz de dados de forma desnecessária, aumentando drasticamente o tempo de treino sem contrapartida em performance preditiva.
* **`COMUNINF` (Município de Provável Infecção):** Esta variável foi descartada por dois motivos críticos:
  1. **Alto Índice de Dados Ausentes:** Apresentava **37,63% de missing values**, o que comprometeria a qualidade da indução da árvore de decisão ou exigiria técnicas complexas de imputação que injetariam ruído no modelo.
  2. **Irrelevância Clínica para a Triagem:** Sob a ótica médica e de negócio, o local geográfico onde o paciente contraiu a infecção não exerce influência biológica ou causal sobre a gravidade da evolução da doença. O que determina a real necessidade de hospitalização são os sintomas clínicos manifestados no momento do atendimento e o histórico de comorbidades do indivíduo, tornando a coluna dispensável para o poder preditivo do modelo.

Mantivemos na base apenas as colunas principais de localização geográfica de notificação (`ID_MUNICIP` e `SG_UF_NOT`) como as representantes oficiais do posicionamento espacial do caso no momento da entrada no sistema.

In [14]:
df["SG_UF_NOT"] = df["SG_UF_NOT"].str.replace("sp", "35")

print("---------- Município de Notificação == Residência  ---------")
print((df["ID_MUNICIP"] == df["ID_MN_RESI"]).value_counts())
print("------------------------------------------------------------\n")

print("------------- UF de Notificação == Residência  -------------")
print((df["SG_UF_NOT"].astype(str) == df["SG_UF"].astype(str)).value_counts())
print("------------------------------------------------------------\n")

print("----------- Município de Notificação == Infecção  ----------")
print((df["ID_MUNICIP"] == df["COMUNINF"]).value_counts())
print("------------------------------------------------------------\n")

print("------------ Município de Infecção mais comuns  ------------")
print(df["COMUNINF"].value_counts().head())
print("------------------------------------------------------------\n")

df = df.drop(columns=["ID_MN_RESI", "SG_UF", "COMUNINF"])

---------- Município de Notificação == Residência  ---------
True     869185
False    102273
Name: count, dtype: int64
------------------------------------------------------------

------------- UF de Notificação == Residência  -------------
True     964218
False      7240
Name: count, dtype: int64
------------------------------------------------------------

----------- Município de Notificação == Infecção  ----------
True     538756
False    432702
Name: count, dtype: int64
------------------------------------------------------------

------------ Município de Infecção mais comuns  ------------
COMUNINF
355030.0    37804
530010.0    14168
420910.0    13504
0.0         13316
350950.0    12468
Name: count, dtype: int64
------------------------------------------------------------



## Normalização da Variável de Idade (`NU_IDADE_N`)

### A Regra de Negócio do SINAN
Para que o algoritmo consiga interpretar corretamente o impacto biológico da idade na evolução da dengue, foi necessário decodificar o atributo com base nas seguintes regras do SINAN:
* **Prefixo 1:** Idade em Horas
* **Prefixo 2:** Idade em Dias
* **Prefixo 3:** Idade em Meses
* **Prefixo 4:** Idade em Anos

### Engenharia de Recursos para Unificação da Escala (`NU_IDADE_N_ANOS`)

A função `transform_ages` foi desenvolvida para decodificar essa estrutura posicional. Todos os registros foram convertidos e normalizados para a unidade de **Anos**, utilizando divisões exatas de tempo para os menores de 1 ano (meses divididos por 12 e dias divididos por 365).

Com a criação do novo atributo contínuo `NU_IDADE_N_ANOS` e o descarte da coluna codificada original, garantiu-se que:
1. **Poder Preditivo Real:** O modelo agora entende perfeitamente os extremos de vulnerabilidade clínica da dengue, separando com precisão os recém-nascidos/bebês (ex: 0.4 anos) e os idosos, que historicamente apresentam maior taxa de hospitalização.
2. **Eliminação de Distorções:** Evitou-se que o algoritmo gerasse nós de decisão errôneos baseados nos códigos brutos, blindando a integridade estatística da Árvore de Decisão.

In [15]:
def transform_ages(age_code):
  age_code_str = str(int(age_code)).zfill(4)

  prefix = age_code_str[0]
  value = float(age_code_str[1:])

  if prefix == "1":
    return value / (24 * 365)
  elif prefix == "2":
    return value / 365
  elif prefix == "3":
    return value / 12
  elif prefix == "4":
    return value
  else:
    return value

df["NU_IDADE_N_ANOS"] = df["NU_IDADE_N"].apply(transform_ages)
df["NU_IDADE_N_ANOS"].describe()
df = df.drop(columns=["NU_IDADE_N"])

## Removendo Exames Laboratoriais Irregulares
Para garantir a qualidade máxima dos rótulos que alimentarão o modelo, realizamos um cruzamento de dados entre a variável `CRITERIO` (que indica como o caso de dengue foi confirmado) e os resultados reais da lista de exames laboratoriais disponíveis (`RESUL_SORO`, `RESUL_NS1`, `RESUL_VI_N`, `RESUL_PCR_`, `HISTOPA_N`, `IMUNOH_N`).

### Diagnóstico de Inconsistências do Sistema (Avaliação Estatística)
De acordo com o manual de operação do SINAN, um caso só pode ser registrado com Critério Laboratorial (`CRITERIO == 1`) se houver comprovação por exames. Ao cruzarmos essas regras via lógica booleana, identificamos uma grave inconsistência operacional na base bruta:

* **Casos Clínicos:** 489.662 registros (Confirmados por sintomas e histórico de surto local).
* **Casos Laboratoriais Legítimos:** 342.589 registros (rotulados como laboratoriais e com pelo menos um exame positivo confirmado na lista).
* **Casos Laboratoriais Inconsistentes (Irregulares):** 103.750 registros. Esses pacientes foram marcados com confirmação laboratorial, mas apresentavam dados vazios ou negativos em absolutamente todos os exames realizados.
* **Casos Ignorados/Não Preenchidos:** 35.457 registros.

### Ação de Saneamento e Readequação de Atributos
Para evitar que o modelo de Machine Learning aprendesse com dados contraditórios ou falsos positivos laboratoriais, aplicamos as seguintes ações de saneamento:

1. **Exclusão de Casos Irregulares:** Removemos completamente os 103.750 registros inconsistentes do DataFrame, mantendo apenas os casos válidos (clínicos, laboratoriais confirmados por exames e os ignorados temporários).
2. **Mapeamento de Alvos e Padronização:** Os casos com critério "Ignorado" (`0` ou `9`) foram convertidos para `NaN` para posterior tratamento de imputação ou descarte seguro.
   * Os casos de critério clínico (`2` ou `3`) foram unificados e remapeados para o identificador numérico **`0.0`**, enquanto os laboratoriais legítimos mantiveram o identificador **`1.0`**.
3. **Redução de Dimensionalidade por Redundância:** Uma vez garantido que os pacientes restantes possuem diagnósticos consistentes, as 6 colunas de resultados de exames específicos (`exams`) foram **descartadas**. Isso foi feito porque o modelo de triagem de entrada não deve prever se o paciente está com dengue (isso já está confirmado), mas sim se ele vai evoluir para a hospitalização.

Após essa filtragem cirúrgica, a coluna `CRITERIO` foi perfeitamente regularizada e estabilizada, deixando a base pronta para a indução da árvore de decisão.

In [ ]:
exams = ["RESUL_SORO", "RESUL_NS1", "RESUL_VI_N", "RESUL_PCR_", "HISTOPA_N", "IMUNOH_N"]

clinic_cases = ((df["CRITERIO"] == 2) | (df["CRITERIO"] == 3))
n_laboratorial_cases = (df["CRITERIO"] == 1).sum()
ignored_cases = ((df["CRITERIO"] == 0) | (df["CRITERIO"] == 9) | (df["CRITERIO"].isna()))
laboratorial_positive_exams = (df[exams] == 1).any(axis=1)
laboratorial_cases_with_exams = ((df["CRITERIO"] == 1) & (laboratorial_positive_exams))
laboratorial_cases_without_exams = ((df["CRITERIO"] == 1) & (~laboratorial_positive_exams))

print(" ------------------ AVALIAÇÃO ESTATÍSTICA --------------------")
print(f"\nQuantidade de casos laboratoriais: {n_laboratorial_cases}.")
print(f"Quantidade de casos laboratoriais com exame realizado: {laboratorial_cases_with_exams.sum()}.")
print(f"Quantidade de casos laboratoriais sem exame realizado: {laboratorial_cases_without_exams.sum()}.")
print(f"Quantidade de casos clínicos: {clinic_cases.sum()}.")
print(f"Quantidade de casos rotulados como 'Ignorado': {ignored_cases.sum()}.\n")

valid_cases = clinic_cases | laboratorial_cases_with_exams | ignored_cases
df = df[valid_cases]
df["CRITERIO"] = df["CRITERIO"].replace([0, 9], np.nan).replace([2, 3], 0)
print(df["CRITERIO"].value_counts(dropna=False))

df = df.drop(columns=exams)

 ------------------ AVALIAÇÃO ESTATÍSTICA --------------------

Quantidade de casos laboratoriais: 446339.
Quantidade de casos laboratoriais com exame realizado: 342589.
Quantidade de casos laboratoriais sem exame realizado: 103750.
Quantidade de casos clínicos: 489662.
Quantidade de casos rotulados como 'Ignorado': 35457.

CRITERIO
0.0    489662
1.0    342589
NaN     35457
Name: count, dtype: int64


## Transformando os Dados Geográficos

A evolução clínica de uma doença negligenciada como a dengue não depende exclusivamente de fatores biológicos do hospedeiro, mas também do contexto socioeconômico e da infraestrutura de saúde do município onde o paciente é assistido. Cidades com piores indicadores de desenvolvimento tendem a registrar diagnósticos tardios, aumentando o risco de hospitalização.

### Integração de Fontes Externas (ONU / IDHM)
Para capturar essa dimensão macroepidemiológica, integramos ao dataset os dados do Índice de Desenvolvimento Humano Municipal (IDHM) da ONU. Foram selecionados três indicadores estratégicos da base censitária mais recente (2010):
* **`idhm_l`:** IDHM Longevidade (reflete as condições gerais de saúde e expectativa de vida locais).
* **`expectativa_vida`:** Expectativa de vida ao nascer no município.
* **`prop_pobreza`:** Proporção de pessoas vulneráveis ou abaixo da linha da pobreza na região.

### Engenharia de Junção e Compatibilização de Chaves (Merge)
O Sistema de Informação de Agravos de Notificação (SINAN) e o Programa das Nações Unidas para o Desenvolvimento (PNUD/ONU) utilizam padrões diferentes para identificar municípios: o SINAN adota o código truncado do IBGE com 6 dígitos (`ID_MUNICIP`), enquanto a ONU utiliza o código completo com 7 dígitos (`id_municipio`).

Para viabilizar a junção sem perda de integridade, estruturamos o seguinte pipeline de dados:
1. **Tradução de Chaves:** Cruzamos a base da ONU (`df_adh`) com uma tabela de mapeamento (`df_convert`) utilizando um *Left Merge* baseado no ID de 7 dígitos.
2. **Filtragem Espacial:** Isolamos os dados do censo e selecionamos apenas as colunas de interesse junto ao código traduzido de 6 dígitos (`id_municipio_6`).
3. **Enriquecimento Final:** Executamos o merge final ligando o nosso DataFrame principal (`df`) à base socioeconômica tratada, utilizando as chaves equivalentes `ID_MUNICIP` e `id_municipio_6`.

### Descarte de Identificadores Categóricos pós-Merge
Após garantir o mapeamento dos índices socioeconômicos para cada linha do dataset, as colunas geográficas e de controle `SG_UF_NOT`, `ID_MUNICIP` e `id_municipio_6` foram **removidas**. 

Manter esses identificadores textuais e numéricos puros na base geraria problemas de alta cardinalidade e *overfitting*, enquanto os novos atributos contínuos importados (`idhm_l`, `expectativa_vida`, `prop_pobreza`) traduzem o real valor preditivo do território de forma limpa e matemática para a Árvore de Decisão.

In [17]:
df_adh = pd.read_csv("dados/ONU_ADH_MUNICIPIOS.csv")
df_convert = pd.read_csv("dados/CONVERSAO_IDS_MUNICIPIOS.csv")

df_adh = df_adh.merge(df_convert, on="id_municipio", how="left")
cols_merge = ["id_municipio_6", "idhm_l", "expectativa_vida", "prop_pobreza"]
df_adh = df_adh.loc[df_adh["ano"] == 2010, cols_merge]

df = df.merge(df_adh, left_on="ID_MUNICIP", right_on="id_municipio_6", how="left")
df = df.drop(columns=["SG_UF_NOT", "ID_MUNICIP", "id_municipio_6"])

## Tratamento e Limpeza das Variáveis Categóricas

Para que a Árvore de Decisão opere de forma eficiente e sem ruídos, é mandatório que as variáveis categóricas e as respostas booleanas (Sim/Não) sigam uma codificação matemática uniforme. As fichas brutas do SINAN utilizam múltiplos padrões numéricos distintos para codificar a presença de sintomas, comorbidades e dados demográficos, necessitando de uma reestruturação.

### Binarização de Respostas Clínicas e de Controle
A técnica de binarização foi aplicada nas variáveis biológicas e de histórico médico:
* **Sintomas e Comorbidades:** Um loop automatizado processou 21 colunas críticas de sinais clínicos (como `FEBRE`, `VOMITO`, `DOR_RETRO`) e condições preexistentes (como `DIABETES`, `HIPERTENSA`, `RENAL`). O padrão original do sistema (`1` para Sim e `2` para Não) foi convertido para a escala padrão de Machine Learning: **`1.0` (Presença do sintoma/comorbidade)** e **`0.0` (Ausência)**, mapeando dados ignorados (`9`) como `NaN`.
* **Variável Alvo (`HOSPITALIZ`):** Seguiu o mesmo tratamento epidemiológico, estabelecendo o mapeamento binário final: **`1.0`** para pacientes internados e **`0.0`** para os não hospitalizados.
* **Perfil de Vulnerabilidade (`CS_SEXO` e `CS_GESTANT`):** O sexo biológico foi mapeado para `0` (Masculino) e `1` (Feminino). Para gestantes, unificamos os diferentes trimestres de gravidez (códigos `1`, `2`, `3` e `4`) no rótulo unificado **`1` (Gestante)**, convertendo as negações e incompatibilidades biológicas para **`0` (Não gestante)**.

### Agrupamento e Redução de Cardinalidade Demográfica (Binning)
Atributos com excesso de categorias distintas (alta cardinalidade) tendem a pulverizar os nós de uma Árvore de Decisão, degradando sua capacidade de generalização. Para mitigar esse problema, aplicamos o mapeamento estruturado:
1. **Dicionário de Raça (`CS_RACA`):** Substituímos os índices numéricos do IBGE pelos seus rótulos textuais explícitos (Branca, Preta, Parda, Amarela, Indígena), convertendo o código de omissão (`9`) em um identificador explícito de ignorado.
2. **Agrupamento de Escolaridade (`CS_ESCOL_N`):** A base original fragmentava o nível de instrução em códigos específicos por séries cursadas. Agrupamos essas categorias redundantes em blocos macro (ex: unificando os códigos de ensino fundamental incompleto parciais em um único rótulo `FUNDAMENTAL_INCOMPLETO`). Isso reduz drasticamente o número de ramificações inúteis que o algoritmo precisaria testar.

Após essa etapa de limpeza e mapeamento, o dataset encontra-se totalmente normalizado do ponto de vista categórico, combinando variáveis binárias puras (`0` ou `1`) e variáveis qualitativas prontas para codificação final.

In [18]:
df["CS_GESTANT"] = df["CS_GESTANT"].replace([2, 3, 4], 1).replace([5, 6], 0).replace(9, np.nan)

df["CS_SEXO"] = df["CS_SEXO"].replace("I", np.nan).replace("M", 0).replace("F", 1)

race_mapping = {
    1: "BRANCA",
    2: "PRETA",
    3: "AMARELA",
    4: "PARDA",
    5: "INDIGENA",
    9: "IGNORADO",
}
df["CS_RACA"] = df["CS_RACA"].replace(race_mapping)

scholarity_mapping = {
    0: "ANALFABETO",
    1: "FUNDAMENTAL_INCOMPLETO",
    2: "FUNDAMENTAL_INCOMPLETO",
    3: "FUNDAMENTAL_INCOMPLETO",
    4: "FUNDAMENTAL_COMPLETO",
    5: "MEDIO_INCOMPLETO",
    6: "MEDIO_COMPLETO",
    7: "SUPERIOR_INCOMPLETO",
    8: "SUPERIOR_COMPLETO",
    9: "IGNORADO",
    10: "MENOR_7_ANOS",
}
df["CS_ESCOL_N"] = df["CS_ESCOL_N"].replace(scholarity_mapping)

cols_clinic_signals = [
    "FEBRE", "MIALGIA", "CEFALEIA", "EXANTEMA", "VOMITO", "NAUSEA", "DOR_COSTAS", "CONJUNTVIT", "ARTRITE", "ARTRALGIA", "PETEQUIA_N", 
    "LEUCOPENIA", "LACO", "DOR_RETRO", "DIABETES", "HEMATOLOG", "HEPATOPAT", "RENAL", "HIPERTENSA", "ACIDO_PEPT", "AUTO_IMUNE"
]
for col in cols_clinic_signals:
    df[col] = df[col].replace(2, 0).replace(9, np.nan)
    
df["HOSPITALIZ"] = df["HOSPITALIZ"].replace(2, 0).replace(9, np.nan)

## Renomeando Atributos

Como etapa final do pipeline de pré-processamento de dados, realizamos uma reestruturação completa nos nomes das colunas remanescentes do dataset. O padrão original do SINAN adota siglas truncadas e codificações operacionais (como `ACIDO_PEPT`, `CS_ESCOL_N` ou `NU_IDADE_N_ANOS`) que prejudicam a legibilidade do código e a posterior interpretação do modelo.

### Objetivos da Renomeação dos Atributos
O mapeamento estruturado por meio do método `.rename()` foi desenhado para garantir:
* **Interpretabilidade Clínica:** Garantir que as variáveis de sintomas (ex: `DOR_RETROORBITAL`, `PETEQUIAS`) e comorbidades (ex: `DOENCAS_HEMATOLOGICAS`, `HIPERTENSAO`) sejam facilmente identificáveis por profissionais da saúde.
* **Transparência na Análise de Importância (*Feature Importance*):** Como o algoritmo escolhido é uma Árvore de Decisão, a plotagem da relevância de cada atributo nos nós de corte gerará gráficos limpos, autoexplicativos e prontos para apresentação.
* **Padronização de Métricas Externas:** Alinhamos os nomes das variáveis importadas do censo da ONU (`IDH_MUNICIPAL_LONGEVIDADE`, `EXPECTATIVA_VIDA`, `PROPORCAO_POBRES`) ao mesmo padrão de caixa alta do restante do ecossistema de dados do projeto.

Após a execução desse dicionário de tradução, a base de dados atinge seu estado definitivo de limpeza e padronização textual.

In [19]:
cols = {
    "NU_ANO": "ANO",
    "CS_SEXO": "SEXO", 
    "CS_GESTANT": "GESTANTE", 
    "CS_RACA": "RACA", 
    "CS_ESCOL_N": "ESCOLARIDADE", 
    "CONJUNTVIT": "CONJUNTIVITE", 
    "PETEQUIA_N": "PETEQUIAS", 
    "DOR_RETRO": "DOR_RETROORBITAL", 
    "HEMATOLOG": "DOENCAS_HEMATOLOGICAS", 
    "HEPATOPAT": "HEPATOPATIAS", 
    "RENAL": "DOENCA_RENAL",
    "HIPERTENSA": "HIPERTENSAO", 
    "ACIDO_PEPT": "DOENCA_ACIDO_PEPTICA", 
    "AUTO_IMUNE": "DOENCAS_AUTO_IMUNE", 
    "HOSPITALIZ": "HOSPITALIZACAO", 
    "CRITERIO": "CRITERIO_CONFIRMACAO",
    "NU_IDADE_N_ANOS": "IDADE", 
    "idhm_l": "IDH_MUNICIPAL_LONGEVIDADE",
    "expectativa_vida": "EXPECTATIVA_VIDA", 
    "prop_pobreza": "PROPORCAO_POBRES"
}
df = df.rename(columns=cols)

## Diagnóstico Final da Base de Dados Consolidada

Após a execução completa do pipeline de engenharia, limpeza e enriquecimento de dados, realizamos uma última auditoria estatística para avaliar a qualidade final da matriz antes de alimentarmos os modelos de Machine Learning.

### Visão Geral do Dataset Resultante
O dataset final foi enxugado para **33 atributos estruturados** (redução drástica a partir das 121 colunas originais). O volume total de registros estabilizou-se em **867.708 linhas**, mantendo uma amostragem massiva e representativa para o aprendizado estatístico.

### Qualidade Estatística e Mitigação de Missing Values
O impacto de todas as etapas de saneamento reflete-se na distribuição final de dados ausentes (*missing values*):
* **Média de Missing por Linha:** Reduzida de **50,16%** para apenas **1,55%**.
* **Mediana de Missing por Linha:** Atingiu o patamar ideal de **0,0%** (indicando que mais da metade de todo o dataset possui preenchimento completo de ponta a ponta).
* **Comportamento por Atributo:** A variável com maior índice de dados ausentes passou a ser a `ESCOLARIDADE` (com 16,1%), seguida por `GESTANTE` (6,6%) e `CRITERIO_CONFIRMACAO` (4,0%). Todas as demais variáveis clínicas de sintomas, comorbidades e indicadores socioeconômicos da ONU encontram-se virtualmente completas, apresentando taxas residuais insignificantes (próximas a 1,1% ou menos).

### Prontidão para Modelagem Preditiva

Com este perfil de dados, garantimos que:
1. **Dados Sem Ruído:** Praticamente eliminamos a necessidade de aplicar técnicas agressivas de imputação estatística (como preenchimento por moda ou média), reduzindo as chances de injetar vieses artificiais no modelo.
2. **Nomes Autoexplicativos:** Todas as colunas agora carregam uma semântica clara e direta em português, facilitando o mapeamento das regras de decisão induzidas pelo algoritmo.
3. **Alvo Intacto:** A variável preditiva `HOSPITALIZACAO` e as chaves de controle temporal não possuem nenhuma perda de informação (0,0% de missing).

O dataset está oficialmente homologado, balanceado e pronto para a etapa de treinamento e validação da Árvore de Decisão (`DecisionTreeClassifier`).

In [20]:
print(len(df.columns))
print(df.columns)

show_missing_stats(df)

33
Index(['ANO', 'SEXO', 'GESTANTE', 'RACA', 'ESCOLARIDADE', 'FEBRE', 'MIALGIA',
       'CEFALEIA', 'EXANTEMA', 'VOMITO', 'NAUSEA', 'DOR_COSTAS',
       'CONJUNTIVITE', 'ARTRITE', 'ARTRALGIA', 'PETEQUIAS', 'LEUCOPENIA',
       'LACO', 'DOR_RETROORBITAL', 'DIABETES', 'DOENCAS_HEMATOLOGICAS',
       'HEPATOPATIAS', 'DOENCA_RENAL', 'HIPERTENSAO', 'DOENCA_ACIDO_PEPTICA',
       'DOENCAS_AUTO_IMUNE', 'HOSPITALIZACAO', 'CRITERIO_CONFIRMACAO',
       'DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO', 'IDADE',
       'IDH_MUNICIPAL_LONGEVIDADE', 'EXPECTATIVA_VIDA', 'PROPORCAO_POBRES'],
      dtype='str')
---------- Missing por Linhas ----------
count    867708.000000
mean          1.558016
std           6.955751
min           0.000000
25%           0.000000
50%           0.000000
75%           3.030303
max          69.696970
dtype: float64
----------------------------------------

---------- Missing por Colunas ---------
ESCOLARIDADE                            0.161840
GESTANTE                           

In [21]:
df.to_csv("dados/BASE.csv", index=False)

# Pré-Processamento dos Dados

## Descrição da Base

A base de dados utilizada neste trabalho é composta por 33 atributos que reúnem informações clínicas, demográficas e socioeconômicas dos pacientes. Para garantir uma correta modelagem e estruturação do pipeline de mineração de dados, as variáveis foram mapeadas e classificadas de acordo com suas propriedades matemáticas, dividindo-se entre os tipos numérico e categórico, além de suas respectivas escalas de mensuração (nominal ou razão) e níveis de cardinalidade (binária, discreta ou contínua), conforme apresentado na tabela a seguir:

| Coluna | Tipo | Escala | Cardinalidade |
| :--- | :---: | :---: | :---: |
| ANO | Numérica | Razão | Discreta |
| SEXO | Categórica | Nominal | Binária |
| GESTANTE | Categórica | Nominal | Binária |
| RACA | Categórica | Nominal | Discreta |
| ESCOLARIDADE | Categórica | Nominal | Discreta |
| FEBRE | Categórica | Nominal | Binária |
| MIALGIA | Categórica | Nominal | Binária |
| CEFALEIA | Categórica | Nominal | Binária |
| EXANTEMA | Categórica | Nominal | Binária |
| VOMITO | Categórica | Nominal | Binária |
| NAUSEA | Categórica | Nominal | Binária |
| DOR_COSTAS | Categórica | Nominal | Binária |
| CONJUNTIVITE | Categórica | Nominal | Binária |
| ARTRITE | Categórica | Nominal | Binária |
| ARTRALGIA | Categórica | Nominal | Binária |
| PETEQUIAS | Categórica | Nominal | Binária |
| LEUCOPENIA | Categórica | Nominal | Binária |
| LACO | Categórica | Nominal | Binária |
| DOR_RETROORBITAL | Categórica | Nominal | Binária |
| DIABETES | Categórica | Nominal | Binária |
| DOENCAS_HEMATOLOGICAS | Categórica | Nominal | Binária |
| HEPATOPATIAS | Categórica | Nominal | Binária |
| DOENCA_RENAL | Categórica | Nominal | Binária |
| HIPERTENSAO | Categórica | Nominal | Binária |
| DOENCA_ACIDO_PEPTICA | Categórica | Nominal | Binária |
| DOENCAS_AUTO_IMUNE | Categórica | Nominal | Binária |
| HOSPITALIZACAO | Categórica | Nominal | Binária |
| CRITERIO_CONFIRMACAO | Categórica | Nominal | Binária |
| DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO | Numérica | Razão | Discreta |
| IDADE | Numérica | Razão | Contínua |
| IDH_MUNICIPAL_LONGEVIDADE | Numérica | Razão | Contínua |
| EXPECTATIVA_VIDA | Numérica | Razão | Contínua |
| PROPORCAO_POBRES | Numérica | Razão | Contínua |

In [22]:
df = pd.read_csv("dados/BASE.csv")
df = df.dropna(subset=["HOSPITALIZACAO"]).copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 867708 entries, 0 to 867707
Data columns (total 33 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   ANO                                   867708 non-null  int64  
 1   SEXO                                  867002 non-null  float64
 2   GESTANTE                              810297 non-null  float64
 3   RACA                                  867704 non-null  str    
 4   ESCOLARIDADE                          727278 non-null  str    
 5   FEBRE                                 857622 non-null  float64
 6   MIALGIA                               857622 non-null  float64
 7   CEFALEIA                              857622 non-null  float64
 8   EXANTEMA                              857622 non-null  float64
 9   VOMITO                                857622 non-null  float64
 10  NAUSEA                                857622 non-null  float64
 11  DOR_COSTAS 

## Análise Estatística dos Atributos

### Atributos Numéricos

| Estatística | DELTA SINTOMAS-NOTIFICAÇÃO | IDADE | IDH MUNICIPAL LONGEVIDADE | EXPECTATIVA DE VIDA | PROPORÇÃO DE POBRES |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Contagem (*count*)** | 315.808 | 315.808 | 315.770 | 315.770 | 315.770 |
| **Média (*mean*)** | 4,72 dias | 35,05 | 0,841 | 75,46 anos | 8,00% |
| **Desvio Padrão (*std*)** | 10,98 dias | 19,76 | 0,029 | 1,73 anos | 9,34% |
| **Mínimo (*min*)** | 0 | 0 | 0,672 | 65,30 anos | 0,00% |
| **25% (1º Quartil)** | 1,00 dia | 20,00 | 0,829 | 74,74 anos | 3,16% |
| **50% (Mediana)** | 3,00 dias | 33,00 | 0,844 | 75,66 anos | 4,59% |
| **75% (3º Quartil)** | 5,00 dias | 49,00 | 0,858 | 76,46 anos | 8,10% |
| **Máximo (*max*)** | 374 dias | 406 | 0,894 | 78,64 anos | 78,59% |

- **Outliers Identificados:** Observa-se que, mesmo após a filtragem da variável alvo, o valor máximo para o atributo `IDADE` consta como **406** e para o delta cronológico consta como **374 dias**. Ambas as métricas representam inconsistências claras de preenchimento ou digitação no sistema original que serão mitigadas pelas quebras da árvore de decisão.

### Atributos Categóricos e Binários (Frequências)

* **ANO DO CASO** (Total: 315.808)
  * 2023: 72.405 registros (~22,9%)
  * 2022: 71.802 registros (~22,7%)
  * 2024: 69.377 registros (~22,0%)
  * 2025: 52.860 registros (~16,7%)
  * 2021: 49.123 registros (~15,6%)
  * 2026: 241 registros (~0,1%)

* **SEXO** (Total válido: 315.515 | Nulos: 293)
  * **1.0 (Feminino / Mapeado):** 170.447 (~54,0%)
  * **0.0 (Masculino / Mapeado):** 145.068 (~46,0%)

* **GESTANTE** (Total válido: 292.675 | Nulos: 23.133)
  * **0.0 (Não):** 290.037 (~99,1%)
  * **1.0 (Sim):** 2.638 (~0,9%)

* **RAÇA / COR** (Total válido: 315.806 | Nulos: 2)
  * BRANCA: 139.559 (~44,2%)
  * PARDA: 111.214 (~35,2%)
  * IGNORADO: 48.018 (~15,2%)
  * PRETA: 12.790 (~4,1%)
  * AMARELA: 3.573 (~1,1%)
  * INDÍGENA: 652 (~0,2%)

* **ESCOLARIDADE** (Total válido: 267.247 | Nulos: 48.561)
  * IGNORADO: 99.968 (~37,4%)
  * MÉDIO COMPLETO: 57.009 (~21,3%)
  * FUNDAMENTAL INCOMPLETO: 36.064 (~13,5%)
  * MENOR 7 ANOS: 21.694 (~8,1%)
  * MÉDIO INCOMPLETO: 17.189 (~6,4%)
  * SUPERIOR COMPLETO: 15.603 (~5,8%)
  * FUNDAMENTAL COMPLETO: 13.236 (~5,0%)
  * SUPERIOR INCOMPLETO: 5.015 (~1,9%)
  * ANALFABETO: 1.469 (~0,6%)

### Manifestações Clínicas (Sintomas)

* **FEBRE**
  * **1.0 (Sim):** 263.485 (~85,6%) | **0.0 (Não):** 44.244 (~14,4%)
* **MIALGIA**
  * **1.0 (Sim):** 245.208 (~79,7%) | **0.0 (Não):** 62.521 (~20,3%)
* **CEFALEIA**
  * **1.0 (Sim):** 244.474 (~79,4%) | **0.0 (Não):** 63.255 (~20,6%)
* **EXANTEMA**
  * **0.0 (Não):** 269.140 (~87,5%) | **1.0 (Sim):** 38.589 (~12,5%)
* **VOMITO**
  * **0.0 (Não):** 230.693 (~75,0%) | **1.0 (Sim):** 77.036 (~25,0%)
* **NAUSEA**
  * **0.0 (Não):** 184.421 (~59,9%) | **1.0 (Sim):** 123.308 (~40,1%)
* **DOR_COSTAS**
  * **0.0 (Não):** 215.945 (~70,2%) | **1.0 (Sim):** 91.784 (~29,8%)
* **CONJUNTIVITE**
  * **0.0 (Não):** 296.271 (~96,3%) | **1.0 (Sim):** 11.458 (~3,7%)
* **ARTRITE**
  * **0.0 (Não):** 276.683 (~89,9%) | **1.0 (Sim):** 31.046 (~10,1%)
* **ARTRALGIA**
  * **0.0 (Não):** 249.129 (~81,0%) | **1.0 (Sim):** 58.600 (~19,0%)
* **PETEQUIAS**
  * **0.0 (Não):** 284.965 (~92,6%) | **1.0 (Sim):** 22.764 (~7,4%)
* **LEUCOPENIA**
  * **0.0 (Não):** 293.275 (~95,3%) | **1.0 (Sim):** 14.454 (~4,7%)
* **LACO (Prova do Laço)**
  * **0.0 (Não):** 297.165 (~96,6%) | **1.0 (Sim):** 10.564 (~3,4%)
* **DOR_RETROORBITAL**
  * **0.0 (Não):** 209.905 (~68,2%) | **1.0 (Sim):** 97.824 (~31,8%)

### Comorbidades e Histórico Clínico

* **DIABETES**
  * **0.0 (Não):** 295.704 (~96,1%) | **1.0 (Sim):** 12.019 (~3,9%)
* **DOENCAS_HEMATOLOGICAS**
  * **0.0 (Não):** 306.214 (~99,5%) | **1.0 (Sim):** 1.510 (~0,5%)
* **HEPATOPATIAS**
  * **0.0 (Não):** 306.177 (~99,5%) | **1.0 (Sim):** 1.547 (~0,5%)
* **DOENCA_RENAL**
  * **0.0 (Não):** 306.311 (~99,5%) | **1.0 (Sim):** 1.411 (~0,5%)
* **HIPERTENSAO**
  * **0.0 (Não):** 279.214 (~90,7%) | **1.0 (Sim):** 28.510 (~9,3%)
* **DOENCA_ACIDO_PEPTICA**
  * **0.0 (Não):** 306.166 (~99,5%) | **1.0 (Sim):** 1.556 (~0,5%)
* **DOENCAS_AUTO_IMUNE**
  * **0.0 (Não):** 305.833 (~99,4%) | **1.0 (Sim):** 1.889 (~0,6%)

### Desfecho e Confirmação

* **HOSPITALIZACAO (Variável Alvo)** (Total: 315.808)
  * **0.0 (Não Internado):** 302.210 (~95,7%)
  * **1.0 (Internado):** 13.598 (~4,3%)

* **CRITERIO_CONFIRMACAO** (Total válido: 305.223 | Nulos: 10.585)
  * **0.0 (Clínico-Epidemiológico / Mapeado):** 198.221 (~64,9%)
  * **1.0 (Laboratorial / Mapeado):** 107.002 (~35,1%)

In [23]:
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]
print(df[numeric_cols].describe(), "\n")

for col in df.columns:
    if col not in numeric_cols:
        print(df[col].value_counts(), "\n")

       DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO          IDADE  \
count                         867708.000000  867708.000000   
mean                               5.317163      37.234815   
std                               11.606705      22.152232   
min                                0.000000       0.000000   
25%                                1.000000      19.000000   
50%                                3.000000      35.000000   
75%                                5.000000      53.000000   
max                              388.000000     408.000000   

       IDH_MUNICIPAL_LONGEVIDADE  EXPECTATIVA_VIDA  PROPORCAO_POBRES  
count              867624.000000     867624.000000     867624.000000  
mean                    0.840398         75.417893          8.502173  
std                     0.029858          1.790600          9.767696  
min                     0.672000         65.300000          0.000000  
25%                     0.828000         74.650000          3.270000  
50%            

## Avaliação dos Resultados dos Processos de Data Mining no Modelo Decision Tree (DT)

In [24]:
def unbalanced_train_test_split(df, test_size, prop_hosp, random_state):
    df_hospitalized = df[df["HOSPITALIZACAO"] == 1]
    df_non_hospitalized = df[df["HOSPITALIZACAO"] == 0]
    
    num_test = int(len(df) * test_size)
    num_hospitalized_test = int(num_test * prop_hosp)
    num_non_hospitalized_test = int(num_test * (1 - prop_hosp))
    
    # Sorteio teste
    X_y_hospitalized_test = df_hospitalized.sample(n=num_hospitalized_test, random_state=random_state)
    X_y_non_hospitalized_test = df_non_hospitalized.sample(n=num_non_hospitalized_test, random_state=random_state)
    X_y_test = pd.concat([X_y_hospitalized_test, X_y_non_hospitalized_test])
    
    # Treino fica com o resto
    X_y_hospitalized_train = df_hospitalized.drop(X_y_hospitalized_test.index)
    X_y_non_hospitalized_train = df_non_hospitalized.drop(X_y_non_hospitalized_test.index)
    X_y_train = pd.concat([X_y_hospitalized_train, X_y_non_hospitalized_train])
    
    X_cols = [col for col in df.columns if col not in ["HOSPITALIZACAO"]]
    X_train, y_train = X_y_train[X_cols], X_y_train["HOSPITALIZACAO"]
    X_test, y_test = X_y_test[X_cols], X_y_test["HOSPITALIZACAO"]
    
    return X_train, X_test, y_train, y_test

In [25]:
X_train, X_test, y_train, y_test = unbalanced_train_test_split(
    df.drop(columns=["ANO"]), test_size=0.2, prop_hosp=0.032190922355718, random_state=42
)

print(y_train.value_counts() / len(y_train))
print(y_test.value_counts() / len(y_test))

HOSPITALIZACAO
0.0    0.514904
1.0    0.485096
Name: count, dtype: float64
HOSPITALIZACAO
0.0    0.967811
1.0    0.032189
Name: count, dtype: float64


In [ ]:
def run_dt_pipeline(df, experiment_name, normalize=False, encode=True):
    # Passar a lista de colunas categóricas para transformar
    numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]
    categorical_cols = [col for col in df.columns if col not in numeric_cols + ["ANO", "HOSPITALIZACAO"]]
    if encode:
        df = pd.get_dummies(
            df, 
            columns=categorical_cols, 
            drop_first=True
        )

    # 1. Separar em treino e teste
    # X = df.drop(columns=["HOSPITALIZACAO", "ANO"])
    # y = df["HOSPITALIZACAO"]
    
    # X_train, X_test, y_train, y_test = train_test_split(
    #     X, y, test_size=0.2, random_state=42, stratify=y
    # )
    X_train, X_test, y_train, y_test = unbalanced_train_test_split(
        df.drop(columns=["ANO"]), test_size=0.2, prop_hosp=0.032190922355718, random_state=42
    )
    
    if normalize:
        scaler = MinMaxScaler()
        
        X_train, X_test = X_train.copy(), X_test.copy()
        X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    # 2. Definir o modelo base
    # dt = DecisionTreeClassifier(class_weight="balanced", random_state=42)
    dt = DecisionTreeClassifier(random_state=42)

    # 3. Definir a grade de hiperparâmetros
    # Os hiperparâmetros mais importantes para controlar complexidade e generalização do modelo costumam ser o max_depth, min_samples_split e min_samples_leaf.
    # O criterion define a medida de qualidade da divisão, com opções como gini e entropy nas versões atuais do scikit-learn
    param_grid = {
        "criterion": ["gini", "entropy"],        # Define a métrica de pureza / cálculo do caos dos nós
        "max_depth": [3, 5, 8],                  # Controla o tamanho da árvore / evita overfitting (IMPORTANTE)
        "min_samples_split": [10, 50],           # Evita divisões em grupos muito pequenos (IMPORTANTE)
        "min_samples_leaf": [5, 20],             # Garante tamanho mínimo de amostras por folha (IMPORTANTE)
        "splitter": ["best"],                    # Avalia a divisão matematicamente ideal
        "class_weight": [{0: 1, 1: 1}, {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 5}] 
    }

    # 4. Grid Search com Cross Validation
    # O F1-Score é superior porque, ao ignorar o sucesso fácil dos verdadeiros negativos, ele é diretamente impactado pelas quedas de Precisão e Recall, forçando o Grid Search a escolher um modelo que minimize os Falsos Negativos sem gerar um número absurdamente caótico de alarmes falsos.
    grid_search = GridSearchCV(
        estimator=dt, param_grid=param_grid, cv=5, scoring="f1", n_jobs=-1, verbose=1 
    )

    # 5. Ajustar o modelo aos dados de treino
    grid_search.fit(X_train, y_train)

    # 6. Melhor modelo encontrado
    best_model = grid_search.best_estimator_

    # 7. Previsões no conjunto de teste
    y_pred = best_model.predict(X_test)

    # 8. Resultados
    print("Melhores hiperparâmetros:")
    print(grid_search.best_params_)

    # cv_results_ guarda o desempenho de todas as combinações avaliadas pelo GridSearchCV
    # results = pd.DataFrame(grid_search.cv_results_)
    # cols = [
    #     "mean_test_score",
    #     "std_test_score",
    #     "param_criterion",
    #     "param_max_depth",
    #     "param_min_samples_split",
    #     "param_min_samples_leaf",
    #     # "param_splitter",
    # ]
    # print(results[cols].sort_values(by="mean_test_score", ascending=False).head(10))

    print("\nMelhor média do F1-Score na validação cruzada:")
    print(grid_search.best_score_)

    print("\nAcurácia no teste:")
    print(accuracy_score(y_test, y_pred))

    print("\nMatriz de confusão Teste:")
    conf_matrix = confusion_matrix(y_test, y_pred)
    conf_matrix_df = pd.DataFrame(
        conf_matrix, 
        index=["Real: Não Hospitalizado", "Real: Hospitalizado"], 
        columns=["Previsto: Não Hospitalizado", "Previsto: Hospitalizado"]
    ).astype(object)
    conf_matrix_df.iloc[0, 0] = f"{conf_matrix[0, 0]} (VN)"
    conf_matrix_df.iloc[0, 1] = f"{conf_matrix[0, 1]} (FP)"
    conf_matrix_df.iloc[1, 0] = f"{conf_matrix[1, 0]} (FN)"
    conf_matrix_df.iloc[1, 1] = f"{conf_matrix[1, 1]} (VP)"
    print(conf_matrix_df)

    print("\nRelatório de classificação Teste:")
    print(classification_report(y_test, y_pred, target_names=["Não Hospitalizado", "Hospitalizado"]))

    # 9. Visualização gráfica da melhor árvore
    plt.figure(figsize=(25, 10))
    plot_tree(
        best_model,
        feature_names=X_train.columns,
        class_names=["Não Hospitalizado", "Hospitalizado"],
        filled=True,
        rounded=True,
        fontsize=12,
    )
    plt.title(f"Melhor Árvore de Decisão Encontrada pelo GridSearchCV - {experiment_name}")
    plt.tight_layout()
    plt.show()

    # 11. Visualização textual das regras da árvore
    # rules = export_text(best_model, feature_names=list(X.columns))
    # print("\nRegras da melhor árvore:\n")
    # print(rules)
    
    return best_model

### Modelo na Base 0

In [ ]:
run_dt_pipeline(df, "Base 0")

### Modelo na Base 1 (Tratamento de Outliers e Missing Values)

A opção por realizar o tratamento de outliers numéricos e a imputação / filtragem de missing values de forma unificada na Base 1 justifica-se pelo fato de que ambas as inconsistências estão interligadas no contexto de registros hospitalares reais. Tratar tais fenômenos de forma isolada poderia mascarar vieses ou induzir a árvore de decisão ao overfitting baseado em dados corrompidos. A abordagem conjunta garantiu a integridade estatística do dataset, resultando em um ganho expressivo de Recall (de 47% para 62%) na classe de interesse (Hospitalizados).

#### Tratamento de Outliers

In [ ]:
# Outliers de colunas numéricas
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, y=col, ax=axes[i], color="skyblue")
    axes[i].set_title(f"Distribuição de {col} (Detectando Outliers)")
    axes[i].set_ylabel("")
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

# Variáveis Municipais (IDH, Expectativa de Vida, Proporção de Pobres) nao mexer pq nao representam erro, mas sim a desigualdade real entre os municipios dos estados
# Tratar outliers DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO e IDADE que sao erros
mask_numerical = (df["IDADE"] >= 0) & (df["IDADE"] <= 120) \
                & (df["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"] >= 0) & (df["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"] <= 30)

# Outliers de colunas categóricas
categorical_cols = [col for col in df.columns if col not in numeric_cols + ["HOSPITALIZACAO", "ANO"]]

fig, axes = plt.subplots(9, 3, figsize=(22, 44))
axes = axes.flatten()
for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, ax=axes[i], stat="percent", order=df[col].value_counts().index, palette="pastel", hue=col)
    axes[i].set_title(f"Distribuição por {col} (Detectando Outliers)")
    axes[i].tick_params(axis="x", rotation=45)
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()
# n precisa remover nada, nada eh outlier apenas falta de representatividade

lines_before = len(df)
df_1 = df[mask_numerical]
lines_after = len(df_1)
lines_removed = lines_after - lines_before
print(" ------------------ RELATÓRIO DA LIMPEZA DE OUTLIERS --------------------")
print(f"Linhas antes do filtro: {lines_before}")
print(f"Linhas após o filtro:  {lines_after}")
print(f"Total de outliers removidos: {lines_removed} ({(lines_removed / lines_before) * 100 :.3f}%)")

#### Tratamento de Missing Values

In [ ]:
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]
comorbidities_cols = [
    "GESTANTE", "FEBRE", "MIALGIA", "CEFALEIA", "EXANTEMA", "VOMITO", "NAUSEA", "DOR_COSTAS",
    "CONJUNTIVITE", "ARTRITE", "ARTRALGIA", "PETEQUIAS", "LEUCOPENIA", "LACO", 
    "DOR_RETROORBITAL", "DIABETES", "DOENCAS_HEMATOLOGICAS", "HEPATOPATIAS", 
    "DOENCA_RENAL", "HIPERTENSAO", "DOENCA_ACIDO_PEPTICA", "DOENCAS_AUTO_IMUNE"
]
categorical_cols = ["SEXO", "RACA", "ESCOLARIDADE", "CRITERIO_CONFIRMACAO"]

missing_per_row = df_1.isna().sum(axis=1)

plt.figure(figsize=(10, 6))
sns.histplot(missing_per_row, stat="percent", binwidth=1, color="teal", edgecolor="black", kde=False)
plt.title("Distribuição de Dados Ausentes por Registro (Paciente)", fontsize=14)
plt.xlabel("Quantidade de Atributos Missing na Linha", fontsize=12)
plt.ylabel("Quantidade de Registros (Linhas)", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

lines_before = len(df_1)
df_1 = df_1[missing_per_row < 15] # Mantendo apenas registros que possuem MENOS de 15 campos nulos

df_1[comorbidities_cols] = df_1[comorbidities_cols].fillna(0) # assumir que se n tem eh 0
df_1[categorical_cols] = df_1[categorical_cols].fillna("IGNORADO") # Nas categóricas, o que sobrou vira "IGNORADO"

# Nas numéricas, usamos a mediana
for col in  numeric_cols: 
    if df_1[col].isna().sum() > 0:
        df_1[col] = df_1[col].fillna(df_1[col].median())
lines_after = len(df_1)
lines_removed = lines_after - lines_before
print(" ------------------ RELATÓRIO DA LIMPEZA DE MISSING VALUES --------------------")
print(f"Linhas antes do filtro: {lines_before}")
print(f"Linhas após o filtro:  {lines_after}")
print(f"Total de linhas removidas: {lines_removed} ({(lines_removed / lines_before) * 100 :.3f}%)")

#### Avaliando o Resultado

In [ ]:
run_dt_pipeline(df_1, "Base 1 - Limpeza de Dados (Outliers e Missing Values)")

# A escolha da Base 1 (dados tratados) em detrimento da Base 0 (dados brutos/sujos) justifica-se pelo princípio da generalização e robustez metodológica. Embora a Base 0 apresente métricas de validação ligeiramente superiores em termos absolutos, tal fenômeno é decorrente do overfitting ao ruído, onde o algoritmo de árvore de decisão passa a codificar outliers e padrões arbitrários de dados ausentes (missing values) como regras preditivas validáveis

# Ao realizar a filtragem de registros severamente corrompidos e padronizar a imputação de sintomas e comorbidades de acordo com a semântica epidemiológica (assumindo a ausência do sintoma na falta de registro positivo), a Base 1 elimina esses sinais espúrios. O modelo resultante, portanto, reflete relações clínicas reais e sustentáveis, garantindo que o classificador mantenha sua capacidade preditiva estável e explicável quando submetido a cenários de produção no mundo real, livre dos vícios estatísticos presentes na base original.

df_1.to_csv("dados/BASE_v2.csv", index=False)

### Modelo na Base 2 (Normalização / Transformação)

In [ ]:
df_1 = pd.read_csv("dados/BASE_v2.csv")
df_2 = df_1.copy()
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(df_1[col], kde=True, ax=axes[i], color="teal")
    axes[i].set_title(f"Distribuição de {col}")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Frequência")
axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

# n segue gaussiana, entao usaremos MinMax! implementacao foi feita na funcao run_dt_pipeline()

In [ ]:
run_dt_pipeline(df_2, "Base 2 - Normalização / Transformação", normalize=True)
# nao muda nada para a base 1

### Modelo na Base 3 (Discretização)

In [ ]:
df_1 = pd.read_csv("dados/BASE_v2.csv")
df_3 = df_1.copy()

age_bins = [-1, 12, 60, df_1["IDADE"].max()]
age_labels = ["CRIANCA_ADOLESCENTE", "ADULTO", "IDOSO"]
df_3["IDADE_DISCRETA"] = pd.cut(df_3["IDADE"], bins=age_bins, labels=age_labels)

delta_bins = [-1, 6, 12, 18, 30]
delta_labels = ["NOTIFICACAO_RAPIDA", "NOTIFICACAO_MEDIA", "NOTIFICACAO_TARDIA", "NOTIFICACAO_EXTREMA"]
df_3["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO_DISCRETA"] = pd.cut(df_3["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO"], bins=delta_bins, labels=delta_labels)

socioeconomic_labels = ["Q1_BAIXO", "Q2_MEDIO_BAIXO", "Q3_MEDIO_ALTO", "Q4_ALTO"]
df_3["IDH_MUNICIPAL_LONGEVIDADE_DISCRETA"] = pd.qcut(df_3["IDH_MUNICIPAL_LONGEVIDADE"], q=4, labels=socioeconomic_labels)
df_3["EXPECTATIVA_VIDA_DISCRETA"] = pd.qcut(df_3["EXPECTATIVA_VIDA"], q=4, labels=socioeconomic_labels)
df_3["PROPORCAO_POBRES_DISCRETA"] = pd.qcut(df_3["PROPORCAO_POBRES"], q=4, labels=socioeconomic_labels)

df_3 = df_3.drop(columns=["IDADE", "DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDH_MUNICIPAL_LONGEVIDADE", 
                          "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"])

In [ ]:
run_dt_pipeline(df_3, "Base 3 - Discretização")

# Os resultados obtidos na Base 3 evidenciam o impacto teórico da perda de granularidade dos dados no algoritmo de Árvore de Decisão. Ao discretizar variáveis contínuas cruciais (como IDADE e as métricas socioeconômicas) em intervalos fixos e quantis, limitou-se a capacidade do classificador de encontrar limiares ótimos e específicos de corte, resultando em uma redução sutil do F1-Score na validação cruzada (de 0.162 para 0.156) e no aumento de Falsos Positivos.

# Contudo, a Base 3 cumpre um papel metodológico fundamental: ela simplifica o espaço de estados do modelo, gerando uma estrutura 100% categórica que prioriza a interpretabilidade clínica. Em termos práticos de saúde pública, embora o desempenho estatístico bruto seja ligeiramente inferior ao da Base 1, as regras geradas na Base 3 tornam-se semanticamente inteligíveis para auditores humanos e gestores hospitalares, demonstrando o trade-off clássico entre a precisão matemática fina e a explicabilidade do modelo.

## Análise de Seleção de Variáveis 

### Feature Selection (Variáveis Preditoras x Variáveis Preditoras)

In [ ]:
df_1 = pd.read_csv("dados/BASE_v2.csv")
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "EXPECTATIVA_VIDA", "PROPORCAO_POBRES"]
X = df_1.drop(columns=["ANO", "HOSPITALIZACAO"])

corr_matrix = X[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Matriz de Correlação X com X (Variáveis Numéricas)")
plt.show()

df_1 = df_1.drop(columns=["EXPECTATIVA_VIDA"])
# removeremos a variavel numerica EXPECTATIVA_VIDA

# Em resumo: Por que gastar energia com as numéricas e não com as categóricas? Nas categóricas: A redundância é parcial, o volume de cruzamentos é colossal (1.521 pares) e o nosso método embutido (a Árvore) já faz o trabalho sujo de ignorar as redundantes de forma 100% automatizada e otimizada matematicamente através da métrica de Gini.

### Feature Selection (Variáveis Preditoras x Variável Alvo)

Usou-se Chi-Quadrado, ANOVA F-Value e Feature Importance da Decision Tree

In [ ]:
numeric_cols = ["DELTA_PRIMEIROS_SINTOMAS_NOTIFICACAO", "IDADE", "IDH_MUNICIPAL_LONGEVIDADE", "PROPORCAO_POBRES"]
df_1 = pd.get_dummies(
    df_1, 
    columns=[col for col in df_1.columns if col not in numeric_cols + ["ANO", "HOSPITALIZACAO"]], 
    drop_first=True
)
categorical_cols = [col for col in df_1.columns if col not in numeric_cols + ["ANO", "HOSPITALIZACAO"]]

X = df_1.drop(columns=["ANO", "HOSPITALIZACAO"])
y = df_1["HOSPITALIZACAO"]

# Chi_Quadrado (Categóricas)
df_categorical = pd.DataFrame(index=categorical_cols)
chi2_scores, _ = chi2(X[categorical_cols], y)
df_categorical["CHI2_SCORE"] = chi2_scores
df_categorical["RANKING_ESTATISTICO"] = df_categorical["CHI2_SCORE"].rank(ascending=False, method="min").astype(int)

# ANOVA F-Value (Numéricas)
df_numeric = pd.DataFrame(index=numeric_cols)
anova_scores, _ = f_classif(X[numeric_cols], y)
df_numeric["ANOVA_SCORE"] = anova_scores
df_numeric["RANKING_ESTATISTICO"] = df_numeric["ANOVA_SCORE"].rank(ascending=False, method="min").astype(int)

# Concatenação dos Rankings
df_ranking = pd.concat([df_categorical[["RANKING_ESTATISTICO"]], df_numeric[["RANKING_ESTATISTICO"]]])

# Feature Importance da Decision Tree (Categóricas e Numéricas)
best_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=8,
    min_samples_leaf=20,
    min_samples_split=50,
    splitter="best",
    random_state=42
)
best_tree.fit(X, y)

df_tree = pd.DataFrame(index=X.columns)
df_tree["TREE_IMPORTANCE"] = best_tree.feature_importances_
df_tree["RANKING_TREE_IMPORTANCE"] = df_tree["TREE_IMPORTANCE"].rank(ascending=False, method="min").astype(int)

df_ranking = df_ranking.join(df_tree)
df_ranking["RANKING"] = df_ranking[["RANKING_ESTATISTICO", "RANKING_TREE_IMPORTANCE"]].mean(axis=1).rank(method="min").astype(int)
df_ranking = df_ranking.sort_values(by="RANKING")

df_ranking[["RANKING", "RANKING_ESTATISTICO", "RANKING_TREE_IMPORTANCE", "TREE_IMPORTANCE"]]

In [ ]:
num_cols= 15
top_features = df_ranking.head(num_cols).index.tolist()
df_4 = df_1[top_features + ["ANO", "HOSPITALIZACAO"]]

run_dt_pipeline(df_4, f"Base 4 - Feature Selection ({num_cols} Variáveis)", encode=False)
# Optou-se pela utilização da Base 4 (15 variáveis) em detrimento da Base 1 (todas as variáveis) porque ambos os cenários apresentaram um desempenho preditivo praticamente idêntico no teste (mesmo F1-Score de 0,17 e Recall de 61%).
# Pautando-se pelo princípio da parcimônia, a Base 4 é significativamente superior por dois motivos práticos: melhoria no desempenho computacional (ao eliminar dezenas de colunas que geravam ruído e redundância matemática) e viabilidade clínica. Em um cenário real de pronto-socorro, o modelo reduz drasticamente a quantidade de campos necessários para o médico preencher na triagem, tornando o sistema muito mais ágil, rápido e aplicável à rotina hospitalar sem perda de qualidade nos diagnósticos.

df_4.to_csv("dados/BASE_v3.csv", index=False)